In [24]:
import os
import json
import time
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    DecisionTreeClassifier, RandomForestClassifier,
    LogisticRegression, 
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array

import pandas as pd

### Configuration & Spark session

In [2]:
PROJECT_ROOT = Path(
    r"D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics"
).resolve()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "output"))
FEATURE_PARQUET_PATH = str(OUTPUT_DIR / "final_feature_dataset.parquet")
FEATURE_METADATA_PATH = str(OUTPUT_DIR / "feature_metadata.json")
BEST_MODEL_PATH = str(OUTPUT_DIR / "best_model")

N_CORES = int(os.environ.get("SPARK_CORES", "8"))
CV_FOLDS = int(os.environ.get("CV_FOLDS", "3"))
SEED = 42

spark = (
    SparkSession.builder
    .appName("BusRoute_05_MachineLearning")
    .master(f"local[{N_CORES}]")
    .config("spark.sql.shuffle.partitions", str(N_CORES * 2))
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "6g"))
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI     :", spark.sparkContext.uiWebUrl)

Spark version: 3.5.8
Spark UI     : http://DESKTOP-P7PE4MO:4041


### Load engineered dataset and its feature-role manifest

In [3]:
schedule = spark.read.parquet(FEATURE_PARQUET_PATH).cache()
BASE_ROW_COUNT = schedule.count()

with open(FEATURE_METADATA_PATH) as f:
    feature_metadata = json.load(f)

print("="*60)
print("Dataset Loaded")
print("="*60)
print("Rows    :", f"{BASE_ROW_COUNT:,}")
print("Columns :", len(schedule.columns))
print()
print("Feature metadata from Notebook 04:")
print(json.dumps(feature_metadata, indent=2))

assert BASE_ROW_COUNT > 0, "final_feature_dataset.parquet loaded with zero rows."

Dataset Loaded
Rows    : 926,481
Columns : 32

Feature metadata from Notebook 04:
{
  "target_column": "route_popularity",
  "leakage_columns": [
    "daily_journeys"
  ],
  "removed_leakage_features_at_source": [
    "service_category"
  ],
  "id_columns": [
    "source_file",
    "vehicle_journey_code",
    "stop_point_ref",
    "line_ref",
    "line_name",
    "service_ref",
    "journey_pattern_ref",
    "stop_name",
    "operator_ref"
  ],
  "timestamp_columns": [
    "scheduled_ts"
  ],
  "candidate_feature_columns": [
    "fare_publication_count",
    "fare_publication_level",
    "fare_status",
    "hour_of_day",
    "is_first_stop",
    "is_peak_hour",
    "journey_type",
    "operator_routes",
    "operator_size",
    "operator_trip_count",
    "operator_workload",
    "route_complexity",
    "scheduled_time",
    "stop_activity",
    "stop_busyness",
    "stop_position",
    "stop_progress_pct",
    "stop_sequence",
    "total_stops",
    "unique_stops"
  ],
  "dropped_const

## Pre-training validation

In [4]:
TARGET_COLUMN = feature_metadata["target_column"]
LEAKAGE_COLUMNS = feature_metadata["leakage_columns"]
ID_COLUMNS = feature_metadata["id_columns"]
TIMESTAMP_COLUMNS = feature_metadata["timestamp_columns"]

candidate_features = [
    c for c in feature_metadata["candidate_feature_columns"]
    if c in schedule.columns
]
missing_from_df = set(feature_metadata["candidate_feature_columns"]) - set(schedule.columns)
if missing_from_df:
    print(f"Note: manifest lists columns not present in this parquet file (already dropped?): {missing_from_df}")

DISPLAY_ID_COLUMNS = [c for c in ["line_ref", "vehicle_journey_code", "stop_point_ref"]
                       if c in schedule.columns and c not in candidate_features]

model_df = schedule.select([TARGET_COLUMN] + DISPLAY_ID_COLUMNS + candidate_features)

print(f"Target column     : {TARGET_COLUMN}")
print(f"Excluded (leakage): {LEAKAGE_COLUMNS}")
print(f"Excluded (id)     : {ID_COLUMNS}")
print(f"Excluded (ts)     : {TIMESTAMP_COLUMNS}")
print(f"\nCandidate features ({len(candidate_features)}): {candidate_features}")

assert not (set(candidate_features) & set(LEAKAGE_COLUMNS)), \
    "A leakage column slipped into the feature set -- check the manifest / this cell's filtering logic."
assert "service_category" not in candidate_features, (
    "service_category is target-derived (built from route_popularity) and "
    "must never be a model feature -- re-run the current Notebook 04, which "
    "removes it at the point of creation instead of just tagging it."
)

Target column     : route_popularity
Excluded (leakage): ['daily_journeys']
Excluded (id)     : ['source_file', 'vehicle_journey_code', 'stop_point_ref', 'line_ref', 'line_name', 'service_ref', 'journey_pattern_ref', 'stop_name', 'operator_ref']
Excluded (ts)     : ['scheduled_ts']

Candidate features (20): ['fare_publication_count', 'fare_publication_level', 'fare_status', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'journey_type', 'operator_routes', 'operator_size', 'operator_trip_count', 'operator_workload', 'route_complexity', 'scheduled_time', 'stop_activity', 'stop_busyness', 'stop_position', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops']


In [5]:
null_report = model_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in model_df.columns
]).collect()[0].asDict()

null_df = pd.DataFrame(
    [(c, n, round(100 * n / BASE_ROW_COUNT, 2)) for c, n in null_report.items() if n > 0],
    columns=["Column", "Null Count", "Null %"],
)
if len(null_df):
    print(null_df.sort_values("Null Count", ascending=False).to_string(index=False))
    high_null_cols = null_df[null_df["Null %"] > 50]["Column"].tolist()
    assert not high_null_cols, f"Column(s) with >50% nulls should be re-examined before modelling: {high_null_cols}"
else:
    print("No missing values in the modelling columns.")

No missing values in the modelling columns.


In [6]:
numeric_candidates = [f.name for f in model_df.schema.fields
                       if f.name in candidate_features and isinstance(f.dataType, NumericType)]
categorical_candidates = [c for c in candidate_features if c not in numeric_candidates]

variance_report = model_df.select([F.variance(F.col(c)).alias(c) for c in numeric_candidates]).collect()[0].asDict()
zero_variance = [c for c, v in variance_report.items() if v is None or v == 0]

distinct_report = model_df.select([F.countDistinct(F.col(c)).alias(c) for c in categorical_candidates]).collect()[0].asDict()
constant_categorical = [c for c, n in distinct_report.items() if n <= 1]

print("Zero-variance numeric features:", zero_variance or "none")
print("Constant categorical features :", constant_categorical or "none")

for c in zero_variance + constant_categorical:
    candidate_features.remove(c)
    if c in numeric_candidates:
        numeric_candidates.remove(c)
    if c in categorical_candidates:
        categorical_candidates.remove(c)

print(f"\nNumeric features    ({len(numeric_candidates)}): {numeric_candidates}")
print(f"Categorical features ({len(categorical_candidates)}): {categorical_candidates}")

Zero-variance numeric features: none
Constant categorical features : none

Numeric features    (11): ['fare_publication_count', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'operator_routes', 'operator_trip_count', 'stop_activity', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops']
Categorical features (9): ['fare_publication_level', 'fare_status', 'journey_type', 'operator_size', 'operator_workload', 'route_complexity', 'scheduled_time', 'stop_busyness', 'stop_position']


### Correlation among numeric features 

In [7]:
from itertools import combinations

corr_rows = []
for a, b in combinations(numeric_candidates, 2):
    corr_rows.append((a, b, model_df.stat.corr(a, b)))
corr_df = pd.DataFrame(corr_rows, columns=["Feature A", "Feature B", "Pearson r"]).sort_values(
    "Pearson r", key=abs, ascending=False
)
print(corr_df.to_string(index=False))

highly_correlated = corr_df[corr_df["Pearson r"].abs() >= 0.95]
if len(highly_correlated):
    print("\n! Highly correlated pairs (|r| >= 0.95) -- consider dropping one of each pair:")
    print(highly_correlated.to_string(index=False))
else:
    print("\nNo pair of numeric features exceeds |r| = 0.95.")

             Feature A           Feature B  Pearson r
           total_stops        unique_stops   0.876777
     stop_progress_pct       stop_sequence   0.748699
fare_publication_count operator_trip_count   0.701493
         stop_sequence         total_stops   0.567833
         stop_sequence        unique_stops   0.498054
       operator_routes operator_trip_count   0.487769
fare_publication_count     operator_routes   0.374470
fare_publication_count       stop_activity   0.272244
         is_first_stop   stop_progress_pct  -0.266280
   operator_trip_count       stop_activity   0.246966
         is_first_stop       stop_sequence  -0.218035
         stop_activity        unique_stops  -0.197390
       operator_routes        unique_stops   0.180851
         stop_activity         total_stops  -0.162311
       operator_routes         total_stops   0.148229
         is_first_stop         total_stops  -0.094317
       operator_routes       stop_activity   0.088161
       operator_routes      

In [8]:
class_counts = model_df.groupBy(TARGET_COLUMN).count().orderBy(F.desc("count")).toPandas()
class_counts["pct"] = (100 * class_counts["count"] / BASE_ROW_COUNT).round(2)
print(class_counts.to_string(index=False))

imbalance_ratio = class_counts["count"].max() / class_counts["count"].min()
print(f"\nImbalance ratio (largest class / smallest class): {imbalance_ratio:.2f}")
if imbalance_ratio > 3:
    print("Ratio > 3 -- class weighting will be applied where the estimator supports it.")

route_popularity  count   pct
            High 325492 35.13
             Low 304386 32.85
          Medium 296603 32.01

Imbalance ratio (largest class / smallest class): 1.10


##  Train/test split

In [9]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=SEED)
train_df = train_df.cache()
test_df = test_df.cache()

n_train, n_test = train_df.count(), test_df.count()
print(f"Train rows: {n_train:,} ({100*n_train/BASE_ROW_COUNT:.1f}%)")
print(f"Test rows : {n_test:,} ({100*n_test/BASE_ROW_COUNT:.1f}%)")
assert n_train + n_test == BASE_ROW_COUNT

print("\nTrain class distribution:")
train_df.groupBy(TARGET_COLUMN).count().orderBy(F.desc("count")).show()
print("Test class distribution:")
test_df.groupBy(TARGET_COLUMN).count().orderBy(F.desc("count")).show()

Train rows: 741,465 (80.0%)
Test rows : 185,016 (20.0%)

Train class distribution:
+----------------+------+
|route_popularity| count|
+----------------+------+
|            High|260651|
|             Low|243274|
|          Medium|237540|
+----------------+------+

Test class distribution:
+----------------+-----+
|route_popularity|count|
+----------------+-----+
|            High|64841|
|             Low|61112|
|          Medium|59063|
+----------------+-----+



In [10]:
train_class_counts = train_df.groupBy(TARGET_COLUMN).count().collect()
n_classes = len(train_class_counts)
total = sum(r["count"] for r in train_class_counts)
class_weight_map = {r[TARGET_COLUMN]: total / (n_classes * r["count"]) for r in train_class_counts}
print("Class weights (inverse frequency):", class_weight_map)

weight_expr = F.create_map([F.lit(x) for pair in class_weight_map.items() for x in pair])
train_df = train_df.withColumn("class_weight", weight_expr[F.col(TARGET_COLUMN)])
test_df = test_df.withColumn("class_weight", F.lit(1.0))  # weighting only applies at training time

train_df = train_df.cache()
train_df.select(TARGET_COLUMN, "class_weight").distinct().show()

Class weights (inverse frequency): {'High': 0.9482219519587495, 'Medium': 1.0404773932811315, 'Low': 1.0159532050280753}
+----------------+------------------+
|route_popularity|      class_weight|
+----------------+------------------+
|             Low|1.0159532050280753|
|            High|0.9482219519587495|
|          Medium|1.0404773932811315|
+----------------+------------------+



## Shared pipeline stages (indexing, encoding, assembly, scaling)

In [11]:
label_indexer = StringIndexer(inputCol=TARGET_COLUMN, outputCol="label", handleInvalid="keep")

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_candidates
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_candidates
]

assembler_inputs = numeric_candidates + [f"{c}_ohe" for c in categorical_candidates]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw", handleInvalid="keep")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True)

shared_stages = [label_indexer] + indexers + encoders + [assembler, scaler]
print(f"Shared pipeline stages: {len(shared_stages)}")
print(f"Assembler input columns ({len(assembler_inputs)}): {assembler_inputs}")

Shared pipeline stages: 21
Assembler input columns (20): ['fare_publication_count', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'operator_routes', 'operator_trip_count', 'stop_activity', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops', 'fare_publication_level_ohe', 'fare_status_ohe', 'journey_type_ohe', 'operator_size_ohe', 'operator_workload_ohe', 'route_complexity_ohe', 'scheduled_time_ohe', 'stop_busyness_ohe', 'stop_position_ohe']


In [12]:
def with_weight_if_supported(estimator, weight_col="class_weight"):
    if estimator.hasParam("weightCol"):
        estimator = estimator.setWeightCol(weight_col)
        print(f"  {type(estimator).__name__}: weightCol enabled")
    else:
        print(f"  {type(estimator).__name__}: weightCol not supported by this estimator, skipping")
    return estimator

## Define four models, their param grids, and cross-validators

### Baseline: Logistic Regression (multinomial)

In [13]:
lr = LogisticRegression(featuresCol="features", labelCol="label", predictionCol="prediction", family="multinomial")
lr = with_weight_if_supported(lr)

lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

lr_pipeline = Pipeline(stages=shared_stages + [lr])

  LogisticRegression: weightCol enabled


### Decision Tree

In [14]:
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", predictionCol="prediction", seed=SEED)
dt = with_weight_if_supported(dt)

dt_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10])
    .addGrid(dt.impurity, ["gini", "entropy"])
    .build()
)

dt_pipeline = Pipeline(stages=shared_stages + [dt])

  DecisionTreeClassifier: weightCol enabled


### Random Forest

In [15]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    seed=SEED
)
rf = with_weight_if_supported(rf)

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

rf_pipeline = Pipeline(stages=shared_stages + [rf])

  RandomForestClassifier: weightCol enabled


In [16]:
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

def make_cv(pipeline, grid):
    return CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=grid,
        evaluator=f1_evaluator,
        numFolds=CV_FOLDS,
        parallelism=2,
        seed=SEED,
    )

models = {
    "Logistic Regression (baseline)": make_cv(lr_pipeline, lr_grid),
    "Decision Tree":                  make_cv(dt_pipeline, dt_grid),
    "Random Forest":                  make_cv(rf_pipeline, rf_grid),
}
print(f"{len(models)} models configured for cross-validated training:")
for name in models:
    print(f"  - {name}")

3 models configured for cross-validated training:
  - Logistic Regression (baseline)
  - Decision Tree
  - Random Forest


## Train, tune, and time each model

In [17]:
fitted_models = {}
training_times = {}

for name, cv in models.items():
    print(f"\nTraining: {name} ...")
    start = time.time() 
    fitted_models[name] = cv.fit(train_df)
    elapsed = time.time() - start
    training_times[name] = elapsed
    print(f"  done in {elapsed:.1f}s -- best CV F1: {max(fitted_models[name].avgMetrics):.4f}")


Training: Logistic Regression (baseline) ...
  done in 414.0s -- best CV F1: 0.7175

Training: Decision Tree ...
  done in 614.9s -- best CV F1: 0.9604

Training: Random Forest ...
  done in 1463.2s -- best CV F1: 0.7949


## Evaluate

In [18]:
metric_names = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]
results = []

predictions_by_model = {}
for name, cv_model in fitted_models.items():
    preds = cv_model.transform(test_df).cache()
    predictions_by_model[name] = preds

    row = {"Model": name, "Training Time (s)": round(training_times[name], 1)}
    for metric in metric_names:
        evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric)
        row[metric] = round(evaluator.evaluate(preds), 4)
    results.append(row)

comparison_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print(comparison_df.to_string(index=False))

                         Model  Training Time (s)  accuracy  weightedPrecision  weightedRecall     f1
                 Decision Tree              614.9    0.9610             0.9616          0.9610 0.9609
                 Random Forest             1463.2    0.8073             0.8125          0.8073 0.8045
Logistic Regression (baseline)              414.0    0.7194             0.7183          0.7194 0.7184


### Confusion matrix per model

In [19]:
label_model = fitted_models[list(fitted_models)[0]].bestModel.stages[0]
label_names = label_model.labels
print("Label index -> class name:", dict(enumerate(label_names)))

for name, preds in predictions_by_model.items():
    print(f"\n--- Confusion matrix: {name} ---")
    (
        preds.groupBy("label", "prediction")
        .count()
        .orderBy("label", "prediction")
        .show(n_classes * n_classes)
    )

Label index -> class name: {0: 'High', 1: 'Low', 2: 'Medium'}

--- Confusion matrix: Logistic Regression (baseline) ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|51598|
|  0.0|       1.0| 3207|
|  0.0|       2.0|10036|
|  1.0|       0.0| 7681|
|  1.0|       1.0|44297|
|  1.0|       2.0| 9134|
|  2.0|       0.0|10178|
|  2.0|       1.0|11674|
|  2.0|       2.0|37211|
+-----+----------+-----+


--- Confusion matrix: Decision Tree ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|63915|
|  0.0|       1.0|  724|
|  0.0|       2.0|  202|
|  1.0|       0.0|  990|
|  1.0|       1.0|56304|
|  1.0|       2.0| 3818|
|  2.0|       0.0|  755|
|  2.0|       1.0|  726|
|  2.0|       2.0|57582|
+-----+----------+-----+


--- Confusion matrix: Random Forest ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|61227|
|  0.0|       1.0|  309|
|  0.0|       2.0| 33

### Feature importance

In [27]:
sample_transformed = fitted_models["Decision Tree"].bestModel.transform(train_df.limit(5))
metadata = sample_transformed.schema["features_raw"].metadata
expanded_feature_names = []
attrs = metadata["ml_attr"]["attrs"]

for attr_type in ["numeric", "binary", "nominal"]:
    if attr_type in attrs:
        expanded_feature_names.extend(
            [item["name"] for item in attrs[attr_type]]
        )
print("Expanded feature names:", len(expanded_feature_names))
print(expanded_feature_names[:20])
n_importances = len(fitted_models["Decision Tree"].bestModel.stages[-1].featureImportances)
print(f"assembler_inputs (raw columns)          : {len(assembler_inputs)}")
print(f"expanded_feature_names (vector dimensions): {len(expanded_feature_names)}")
print(f"featureImportances length                : {n_importances}")
assert len(expanded_feature_names) == n_importances, (
    "expanded_feature_names length doesn't match featureImportances length "
    "-- check that all fitted pipelines share the same indexer/encoder/assembler stages."
)

Expanded feature names: 1251
['fare_publication_count', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'operator_routes', 'operator_trip_count', 'stop_activity', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops', 'fare_publication_level_ohe_High', 'fare_publication_level_ohe_Low', 'fare_status_ohe_Available', 'fare_status_ohe_Unavailable', 'journey_type_ohe_Long', 'journey_type_ohe_Medium', 'journey_type_ohe_Short', 'operator_size_ohe_Large', 'operator_size_ohe_Medium']
assembler_inputs (raw columns)          : 20
expanded_feature_names (vector dimensions): 1251
featureImportances length                : 1251


In [28]:
for name in ["Decision Tree", "Random Forest"]:
    best_pipeline_model = fitted_models[name].bestModel
    classifier_stage = best_pipeline_model.stages[-1]
    importances = classifier_stage.featureImportances

    imp_df = pd.DataFrame({
        "feature_name": expanded_feature_names,
        "importance": importances.toArray(),
    }).sort_values("importance", ascending=False).head(15)
    print(f"\nTop 15 features -- {name}:")
    print(imp_df[["feature_name", "importance"]].to_string(index=False))

lr_best_model = fitted_models["Logistic Regression (baseline)"].bestModel
lr_stage = lr_best_model.stages[-1]
coef_matrix = lr_stage.coefficientMatrix.toArray()  # shape: (n_classes, n_features)
print(f"\nLogistic Regression coefficient magnitudes (baseline), by class ({label_names}):")
for class_idx, class_name in enumerate(label_names):
    coef_df = pd.DataFrame({
        "feature_name": expanded_feature_names,
        "coefficient": coef_matrix[class_idx],
    })
    coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
    top = coef_df.sort_values("abs_coefficient", ascending=False).head(10)
    print(f"\n  Class '{class_name}' -- top 10 by |coefficient|:")
    print("  " + top[["feature_name", "coefficient"]].to_string(index=False).replace("\n", "\n  "))


Top 15 features -- Decision Tree:
                  feature_name  importance
                 stop_activity    0.346552
                  unique_stops    0.306254
               operator_routes    0.106000
                   total_stops    0.086837
      route_complexity_ohe_Low    0.060144
       operator_size_ohe_Small    0.024328
                   hour_of_day    0.016220
        fare_publication_count    0.013916
           operator_trip_count    0.011665
      operator_size_ohe_Medium    0.009006
       journey_type_ohe_Medium    0.005279
         journey_type_ohe_Long    0.003522
fare_publication_level_ohe_Low    0.002364
             stop_progress_pct    0.002187
     route_complexity_ohe_High    0.001864

Top 15 features -- Random Forest:
               feature_name  importance
              stop_activity    0.151683
               unique_stops    0.107249
      stop_busyness_ohe_Low    0.097397
     stop_busyness_ohe_High    0.090019
                total_stops    0.071230
  

### Cross-validation results 

In [29]:
for name, cv_model in fitted_models.items():
    print(f"\n--- CV results: {name} ---")
    for params, metric in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics):
        param_str = ", ".join(f"{p.name}={v}" for p, v in params.items())
        print(f"  F1={metric:.4f}  [{param_str}]")
    print(f"  >> best F1: {max(cv_model.avgMetrics):.4f}")


--- CV results: Logistic Regression (baseline) ---
  F1=0.7175  [regParam=0.01, elasticNetParam=0.0]
  F1=0.6557  [regParam=0.01, elasticNetParam=0.5]
  F1=0.6985  [regParam=0.1, elasticNetParam=0.0]
  F1=0.6187  [regParam=0.1, elasticNetParam=0.5]
  >> best F1: 0.7175

--- CV results: Decision Tree ---
  F1=0.7768  [maxDepth=5, impurity=gini]
  F1=0.7758  [maxDepth=5, impurity=entropy]
  F1=0.9529  [maxDepth=10, impurity=gini]
  F1=0.9604  [maxDepth=10, impurity=entropy]
  >> best F1: 0.9604

--- CV results: Random Forest ---
  F1=0.6687  [numTrees=50, maxDepth=5]
  F1=0.7949  [numTrees=50, maxDepth=10]
  F1=0.6597  [numTrees=100, maxDepth=5]
  F1=0.7885  [numTrees=100, maxDepth=10]
  >> best F1: 0.7949


### Cell 23 — Scalability notes (training time vs. model complexity)

In [30]:
scalability_df = comparison_df[["Model", "Training Time (s)", "f1"]].copy()
scalability_df["Time per 0.01 F1"] = (
    scalability_df["Training Time (s)"] / (scalability_df["f1"] * 100)
).round(2)
print(scalability_df.to_string(index=False))
print(
     "\n'Time per 0.01 F1' measures training efficiency. Lower values are "
    "better. Random Forest takes longer to train, so the extra cost should "
    "be justified by a higher F1 score."
)

                         Model  Training Time (s)     f1  Time per 0.01 F1
                 Decision Tree              614.9 0.9609              6.40
                 Random Forest             1463.2 0.8045             18.19
Logistic Regression (baseline)              414.0 0.7184              5.76

'Time per 0.01 F1' measures training efficiency. Lower values are better. Random Forest takes longer to train, so the extra cost should be justified by a higher F1 score.


### Model comparison table and best-model selection

In [31]:
print("="*70)
print("FINAL MODEL COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name].bestModel

baseline_row = comparison_df[comparison_df["Model"] == "Logistic Regression (baseline)"].iloc[0]
best_row = comparison_df.iloc[0]
uplift = best_row["f1"] - baseline_row["f1"]

print(f"\nBaseline (Logistic Regression) F1 : {baseline_row['f1']:.4f}")
print(f"Best model ({best_model_name}) F1{' ' * max(0, 20 - len(best_model_name))}: {best_row['f1']:.4f}")
print(f"Uplift over baseline               : {uplift:+.4f}")

print(f"\nSelected model: {best_model_name}")
print(
    "Best model selected based on F1 score, with accuracy, precision, recall, "
    "and training time also considered. Logistic Regression serves as the "
    "baseline, while Decision Tree and Random Forest are compared to determine "
    "whether their higher complexity provides a worthwhile performance gain."
)

FINAL MODEL COMPARISON
                         Model  Training Time (s)  accuracy  weightedPrecision  weightedRecall     f1
                 Decision Tree              614.9    0.9610             0.9616          0.9610 0.9609
                 Random Forest             1463.2    0.8073             0.8125          0.8073 0.8045
Logistic Regression (baseline)              414.0    0.7194             0.7183          0.7194 0.7184

Baseline (Logistic Regression) F1 : 0.7184
Best model (Decision Tree) F1       : 0.9609
Uplift over baseline               : +0.2425

Selected model: Decision Tree
Best model selected based on F1 score, with accuracy, precision, recall, and training time also considered. Logistic Regression serves as the baseline, while Decision Tree and Random Forest are compared to determine whether their higher complexity provides a worthwhile performance gain.


### Save per-model test predictions 

In [32]:
EVAL_DIR = OUTPUT_DIR / "eval_predictions"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

for name, preds in predictions_by_model.items():
    out_cols = DISPLAY_ID_COLUMNS + [TARGET_COLUMN, "label", "prediction"]
    df_to_save = preds.select(*out_cols, *(
        [vector_to_array(F.col("probability")).alias("probability_array")]
        if "probability" in preds.columns else []
    ))
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    path = EVAL_DIR / f"{safe_name}.parquet"
    df_to_save.write.mode("overwrite").parquet(str(path))
    print(f"Saved predictions -- {name}: {path}")

Saved predictions -- Logistic Regression (baseline): D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\eval_predictions\Logistic_Regression_baseline.parquet
Saved predictions -- Decision Tree: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\eval_predictions\Decision_Tree.parquet
Saved predictions -- Random Forest: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\eval_predictions\Random_Forest.parquet


### Save cross-validation results

In [33]:
cv_results = []
for name, cv_model in fitted_models.items():
    for params, metric in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics):
        cv_results.append({
            "model": name,
            "params": {p.name: v for p, v in params.items()},
            "avg_f1": metric,
        })

with open(OUTPUT_DIR / "cv_results.json", "w") as f:
    json.dump(cv_results, f, indent=2)
print(f"CV results saved to: {OUTPUT_DIR / 'cv_results.json'}")

CV results saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\cv_results.json


### Save feature importances / coefficients and label mapping

In [34]:
feature_importance_export = {
    "assembler_inputs": expanded_feature_names,  
    "label_names": label_names,
    "tree_importances": {},
    "lr_coefficients": {
        class_name: coef_matrix[i].tolist() for i, class_name in enumerate(label_names)
    },
}
for name in ["Decision Tree", "Random Forest"]:
    stage = fitted_models[name].bestModel.stages[-1]
    feature_importance_export["tree_importances"][name] = stage.featureImportances.toArray().tolist()

with open(OUTPUT_DIR / "feature_importance.json", "w") as f:
    json.dump(feature_importance_export, f, indent=2)
print(f"Feature importance / coefficients saved to: {OUTPUT_DIR / 'feature_importance.json'}")

Feature importance / coefficients saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\feature_importance.json


In [35]:
unseen_stand_in_path = OUTPUT_DIR / "unseen_records_sample.parquet"
test_df.drop("class_weight").write.mode("overwrite").parquet(str(unseen_stand_in_path))
print(f"Unseen-records stand-in saved to: {unseen_stand_in_path}")
print(f"Rows: {test_df.count():,}")

Unseen-records stand-in saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\unseen_records_sample.parquet
Rows: 185,016


## Persist best model

In [36]:
best_model.write().overwrite().save(BEST_MODEL_PATH)
print(f"Best model ({best_model_name}) saved to: {BEST_MODEL_PATH}")

comparison_df.to_csv(str(OUTPUT_DIR / "model_comparison.csv"), index=False)
print(f"Model comparison table saved to: {OUTPUT_DIR / 'model_comparison.csv'}")

Best model (Decision Tree) saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\best_model
Model comparison table saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\model_comparison.csv


In [ ]:
spark.stop()
print("Spark session stopped. Notebook 05 complete.")